In [0]:

import pyspark.sql.functions as F

catalog_name = "iranian_transactions"

In [0]:

def build_dim_date(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.dim_date
        USING DELTA AS
        WITH dates AS (
          SELECT explode(sequence(to_date('2024-01-01'), to_date('2025-12-31'), interval 1 day)) AS full_date
        )
        SELECT
          CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_sk,
          full_date,
          year(full_date) AS year,
          quarter(full_date) AS quarter,
          month(full_date) AS month,
          day(full_date) AS day,
          date_format(full_date, 'EEEE') AS day_name,
          date_format(full_date, 'MMMM') AS month_name,
          weekofyear(full_date) AS week_of_year,
          CASE WHEN dayofweek(full_date) IN (1, 7) THEN true ELSE false END AS is_weekend
        FROM dates
    """)
    print("dim_date built")


def build_dim_time(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.dim_time
        USING DELTA AS
        SELECT
          (hour * 100 + minute) AS time_id,
          hour,
          minute,
          CASE
            WHEN hour BETWEEN 5 AND 11  THEN 'Morning'
            WHEN hour BETWEEN 12 AND 16 THEN 'Afternoon'
            WHEN hour BETWEEN 17 AND 20 THEN 'Evening'
            ELSE 'Night'
          END AS time_of_day
        FROM (SELECT explode(sequence(0, 23)) AS hour) h
        CROSS JOIN (SELECT explode(sequence(0, 59)) AS minute) m
    """)
    print("dim_time built")


def build_dim_card(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.dim_card
        USING DELTA AS
        SELECT row_number() OVER (ORDER BY card_provider) AS card_id, card_provider
        FROM (
          SELECT DISTINCT card_type AS card_provider
          FROM {catalog_name}.silver.slv_transactions
          WHERE card_type IS NOT NULL AND TRIM(card_type) != ''
        )
    """)
    print("dim_card built")


def build_dim_location(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.dim_location
        USING DELTA AS
        SELECT row_number() OVER (ORDER BY city_name) AS location_id, city_name
        FROM (
          SELECT DISTINCT city AS city_name
          FROM {catalog_name}.silver.slv_transactions
          WHERE city IS NOT NULL AND TRIM(city) != ''
        )
    """)
    print("dim_location built")


def build_dim_status(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.dim_status
        USING DELTA AS
        SELECT row_number() OVER (ORDER BY status_label) AS status_id, status_label
        FROM (
          SELECT DISTINCT status AS status_label
          FROM {catalog_name}.silver.slv_transactions
          WHERE status IS NOT NULL AND TRIM(status) != ''
        )
    """)
    print("dim_status built")


def build_fact_transactions(spark):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {catalog_name}.gold.fact_transactions
        USING DELTA AS
        SELECT
          t.transaction_key,               -- grain key: content hash from silver, not raw id
          t.source_system,
          t.id                    AS source_id,   -- lineage/audit only -- NOT a key, see silver notes

          d.date_sk,
          tm.time_id,
          c.card_id,
          l.location_id,
          s.status_id,

          t.amount,
          COALESCE(t.fees, 0)            AS fees,
          COALESCE(t.discount, 0)        AS discount,
          COALESCE(t.balance_before, 0)  AS balance_before,
          t.is_debt,
          d.is_weekend,                   -- from dim_date's calendar logic, not the source column
                                           -- (only trx_extra_10k ever populated is_weekend directly)

          t._source_file,
          t.ingested_at
        FROM {catalog_name}.silver.slv_transactions t
        LEFT JOIN {catalog_name}.gold.dim_date     d  ON t.transaction_date = d.full_date
        LEFT JOIN {catalog_name}.gold.dim_time     tm ON t.hour = tm.hour AND t.minute = tm.minute
        LEFT JOIN {catalog_name}.gold.dim_card     c  ON t.card_type = c.card_provider
        LEFT JOIN {catalog_name}.gold.dim_location l  ON t.city = l.city_name
        LEFT JOIN {catalog_name}.gold.dim_status   s  ON t.status = s.status_label
    """)
    print("fact_transactions built")


def validate(spark):
    """Checks that actually exercise the join, not just a count() per table
    with no relationship between the numbers."""
    silver = spark.table(f"{catalog_name}.silver.slv_transactions")
    fact = spark.table(f"{catalog_name}.gold.fact_transactions")

    silver_count = silver.count()
    fact_count = fact.count()
    print(f"\nsilver rows: {silver_count}   fact rows: {fact_count}")
    if silver_count != fact_count:
        raise ValueError(
            "Row count mismatch between silver and fact -- a join is fanning "
            "out (a dimension has duplicate natural-key values) or dropping "
            "rows (an INNER JOIN got used somewhere instead of LEFT JOIN)."
        )

    orphan_checks = {
        "date_sk": "dim_date",
        "time_id": "dim_time",
        "card_id": "dim_card",
        "location_id": "dim_location",
        "status_id": "dim_status",
    }
    print("\nOrphan check (fact rows with no matching dimension row):")
    any_orphans = False
    for fk_col, dim_name in orphan_checks.items():
        orphan_count = fact.filter(F.col(fk_col).isNull()).count()
        flag = "  <-- investigate" if orphan_count else ""
        print(f"  {fk_col:15s} -> {dim_name:14s} orphans: {orphan_count:>6}{flag}")
        any_orphans = any_orphans or orphan_count > 0
    if any_orphans:
        print(
            "  Orphans usually mean a value in silver doesn't match anything "
            "in the dimension -- e.g. a city/card_type/status spelling that "
            "slipped past the silver cleaning maps. Check CITY_MAP / "
            "CARD_TYPE_MAP / STATUS_MAP in silver_transform.py first."
        )

    dupe_count = (
        fact.groupBy("transaction_key").count().filter(F.col("count") > 1).count()
    )
    if dupe_count > 0:
        raise ValueError(
            f"{dupe_count} duplicate transaction_key values in fact_transactions "
            "-- the fact table should be at the same grain as silver."
        )

    print("\nDimension row counts:")
    for name in ["dim_date", "dim_time", "dim_card", "dim_location", "dim_status"]:
        print(f"  {name:14s} {spark.table(f'{catalog_name}.gold.{name}').count()}")


def main():
    build_dim_date(spark)
    build_dim_time(spark)
    build_dim_card(spark)
    build_dim_location(spark)
    build_dim_status(spark)
    build_fact_transactions(spark)
    validate(spark)
    print("\nGold layer rebuilt against unified silver.slv_transactions.")


if __name__ == "__main__":
    main()